In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [ ]:

esquemaProduction = "Production"

dimensionProductCategory = pd.read_sql_table("ProductCategory", motorBaseDatos, esquemaProduction)
dimensionProductCategory


c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


,ProductCategoryID,Name,rowguid,ModifiedDate
0,1,Bikes,cfbda25c-df71-47a7-b81b-64ee161aa37c,2008-04-30
1,2,Components,c657828d-d808-4aba-91a3-af2ce02300e9,2008-04-30
2,3,Clothing,10a7c342-ca82-48d4-8a38-46a2eb089b74,2008-04-30
3,4,Accessories,2be3be36-d9a2-4eee-b593-ed895d97c2a6,2008-04-30


TRANSFORMACION

In [14]:


dimensionProductCategory.rename(columns={
    'ProductCategoryID': 'ProductCategoryKey',
    'Name' : 'EnglishProductCategoryName'
}, inplace=True)

dimensionProductCategory["ProductCategoryAlternateKey"] = dimensionProductCategory["ProductCategoryKey"]
dimensionProductCategory["SpanishProductCategoryName"] = None
dimensionProductCategory["FrenchProductCategoryName"] = None


dimensionProductCategory.drop(columns=[
    'rowguid',
    'ModifiedDate',
], inplace=True)

dimensionProductCategory

,ProductCategoryKey,EnglishProductCategoryName,ProductCategoryAlternateKey,SpanishProductCategoryName,FrenchProductCategoryName
0,1,Bikes,1,None,None
1,2,Components,2,None,None
2,3,Clothing,3,None,None
3,4,Accessories,4,None,None


CARGAR A LA BODEGA

In [15]:
dimensionProductCategory.to_sql('dimensionProductCategory',motorBodegaDatos, if_exists='replace',index=False)

4